# Banking Fraud Detection & Financial Risk Analytics

## Machine Learning — Logistic Regression

### Objective
Detect fraudulent transactions using a time-based evaluation. The original PaySim dataset is synthetic; step is a simulation time index, not a calendar date. Currency is unspecified.

### Methodology
Split by time first; sample only the training period; keep the later test period at its original class distribution. The test period is not used to fit preprocessing or the model.

**Import Libraries and Configure Project Paths**

In [16]:
import os, json, time, joblib
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    classification_report, confusion_matrix,
    average_precision_score, roc_auc_score
)

DATA_PATH = r"C:\Users\Swathi\OneDrive\Desktop\Projects\Banking_Fraud_Analytics\Raw_Data\banking_fraud_analysis_ready.csv"
OUTPUT_DIR = r"C:\Users\Swathi\OneDrive\Desktop\Projects\Banking_Fraud_Analytics\ML_Outputs"

os.makedirs(OUTPUT_DIR, exist_ok=True)

CHUNK_SIZE = 100_000
SAMPLE_FRAC = 0.05
RANDOM_STATE = 42
TARGET = "isFraud"

print("Setup completed!")
print("Dataset exists:", os.path.exists(DATA_PATH))
print("Output folder:", OUTPUT_DIR)

Setup completed!
Dataset exists: True
Output folder: C:\Users\Swathi\OneDrive\Desktop\Projects\Banking_Fraud_Analytics\ML_Outputs


**Calculate the Time Cutoff**

In [17]:
steps = pd.read_csv(DATA_PATH, usecols=["step"])["step"]
time_cutoff = steps.quantile(0.80)
print("Minimum step:", steps.min())
print("Maximum step:", steps.max())
print("80% time cutoff:", time_cutoff)
del steps

Minimum step: 1
Maximum step: 743
80% time cutoff: 355.0


**Training and Testing Data Split**

In [18]:
features = ["step", "type", "amount", "oldbalanceOrg", "oldbalanceDest"]
train_parts, test_parts = [], []
rng = np.random.default_rng(RANDOM_STATE)

for chunk in pd.read_csv(DATA_PATH, chunksize=CHUNK_SIZE):
    train = chunk[chunk["step"] < time_cutoff]
    test = chunk[chunk["step"] >= time_cutoff]

    if not train.empty:
        fraud = train[train[TARGET] == 1]
        normal = train[train[TARGET] == 0].sample(frac=SAMPLE_FRAC, random_state=int(rng.integers(1_000_000)))
        train_parts.append(pd.concat([fraud, normal])[features + [TARGET]])

    if not test.empty:
        test_parts.append(test[features + [TARGET]])

train_df = pd.concat(train_parts, ignore_index=True).sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)
test_df = pd.concat(test_parts, ignore_index=True)

print("Training shape:", train_df.shape)
print("Testing shape:", test_df.shape)
print("\nTraining fraud rate:", round(train_df[TARGET].mean() * 100, 3), "%")
print("Testing fraud rate:", round(test_df[TARGET].mean() * 100, 4), "%")

Training shape: (257214, 6)
Testing shape: (1293523, 6)

Training fraud rate: 1.538 %
Testing fraud rate: 0.3292 %


**Validate Training Data**

In [19]:
print("Training shape:", train_df.shape)
print("\nMissing values:\n", train_df.isnull().sum())
print("\nDuplicate rows:", train_df.duplicated().sum())
print("\nTarget counts:\n", train_df[TARGET].value_counts())
print("\nInvalid target values:", (~train_df[TARGET].isin([0, 1])).sum())
print("Negative amounts:", (train_df["amount"] < 0).sum())

Training shape: (257214, 6)

Missing values:
 step              0
type              0
amount            0
oldbalanceOrg     0
oldbalanceDest    0
isFraud           0
dtype: int64

Duplicate rows: 19

Target counts:
 isFraud
0    253259
1      3955
Name: count, dtype: int64

Invalid target values: 0
Negative amounts: 0


**Separate Features and Target**

In [20]:
X = train_df[features]
y = train_df[TARGET].astype(int)

print("Features shape:", X.shape)
print("Target shape:", y.shape)
print("\nFraud cases:", y.sum())
print("Non-fraud cases:", (y == 0).sum())

Features shape: (257214, 5)
Target shape: (257214,)

Fraud cases: 3955
Non-fraud cases: 253259


**Build the ML Pipeline**

In [21]:
numeric = ["step", "amount", "oldbalanceOrg", "oldbalanceDest"]
categorical = ["type"]

preprocessor = ColumnTransformer([
    ("num", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]), numeric),
    ("cat", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore"))
    ]), categorical)
])

model = Pipeline([
    ("preprocessing", preprocessor),
    ("classifier", LogisticRegression(
        class_weight="balanced",
        max_iter=500,
        random_state=RANDOM_STATE
    ))
])
print("Preprocessing and Logistic Regression pipeline created!")

Preprocessing and Logistic Regression pipeline created!


**Train the Logistic Regression Model**

In [22]:
start = time.time()
model.fit(X, y)
print("Model training completed!")
print("Training time:", round(time.time() - start, 2), "seconds")

Model training completed!
Training time: 2.31 seconds


**Predict on Untouched Test Data**

In [23]:
X_test = test_df[features]
y_test = test_df[TARGET].astype(int)
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]
print("Test records:", len(y_test))
print("Actual fraud cases:", y_test.sum())
print("Predictions completed!")

Test records: 1293523
Actual fraud cases: 4258
Predictions completed!


**Evaluate Model Performance**

In [24]:
print("Classification Report:\n")
print(classification_report(
    y_test, y_pred,
    target_names=["Non-Fraud", "Fraud"],
    digits=4,
    zero_division=0
))
print("Average Precision:", round(average_precision_score(y_test, y_prob), 4))
print("ROC-AUC:", round(roc_auc_score(y_test, y_prob), 4))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))

Classification Report:

              precision    recall  f1-score   support

   Non-Fraud     0.9993    0.9478    0.9729   1289265
       Fraud     0.0476    0.7896    0.0898      4258

    accuracy                         0.9473   1293523
   macro avg     0.5234    0.8687    0.5313   1293523
weighted avg     0.9961    0.9473    0.9700   1293523

Average Precision: 0.3168
ROC-AUC: 0.962

Confusion Matrix:
 [[1222000   67265]
 [    896    3362]]


**Threshold Analysis**

In [25]:
from sklearn.metrics import precision_score, recall_score, f1_score
results = []
for threshold in [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]:
    pred = (y_prob >= threshold).astype(int)
    results.append({
        "Threshold": threshold,
        "Precision": precision_score(y_test, pred, zero_division=0),
        "Recall": recall_score(y_test, pred, zero_division=0),
        "F1_Score": f1_score(y_test, pred, zero_division=0)
    })
threshold_df = pd.DataFrame(results).round(4)
print(threshold_df.to_string(index=False))

 Threshold  Precision  Recall  F1_Score
       0.1     0.0083  0.9937    0.0165
       0.2     0.0098  0.9847    0.0195
       0.3     0.0157  0.9403    0.0309
       0.4     0.0336  0.8567    0.0647
       0.5     0.0476  0.7896    0.0898
       0.6     0.0704  0.7489    0.1287
       0.7     0.1223  0.6595    0.2064
       0.8     0.1933  0.5207    0.2820
       0.9     0.2872  0.4016    0.3349


**Feature Coefficient Analysis**

In [26]:
feature_names = model.named_steps["preprocessing"].get_feature_names_out()
coefficients = model.named_steps["classifier"].coef_[0]
coef_df = pd.DataFrame({"Feature": feature_names,"Coefficient": coefficients})
coef_df["Abs_Coefficient"] = coef_df["Coefficient"].abs()
coef_df = coef_df.sort_values("Abs_Coefficient", ascending=False)
print(coef_df.round(4).to_string(index=False))

            Feature  Coefficient  Abs_Coefficient
  cat__type_CASH_IN     -94.3966          94.3966
 cat__type_TRANSFER      45.9328          45.9328
 cat__type_CASH_OUT      44.4227          44.4227
 num__oldbalanceOrg      14.6002          14.6002
        num__amount      -1.5653           1.5653
  cat__type_PAYMENT      -0.5956           0.5956
num__oldbalanceDest      -0.5433           0.5433
          num__step      -0.1968           0.1968
    cat__type_DEBIT      -0.0078           0.0078


**Save Model and Results**

In [27]:
# Save trained model
joblib.dump(model, os.path.join(OUTPUT_DIR, "fraud_detection_model.joblib"))

# Save evaluation report
report = classification_report(y_test, y_pred,target_names=["Non-Fraud", "Fraud"],output_dict=True,zero_division=0)

with open(os.path.join(OUTPUT_DIR, "classification_report.json"), "w") as f:
    json.dump(report, f, indent=4)

# Save test predictions
predictions = test_df[features + [TARGET]].copy()
predictions["Predicted_Fraud"] = y_pred
predictions["Fraud_Probability"] = y_prob
predictions.to_csv(os.path.join(OUTPUT_DIR, "test_predictions.csv"),index=False)

# Save threshold analysis and coefficients
threshold_df.to_csv(os.path.join(OUTPUT_DIR, "threshold_analysis.csv"), index=False)
coef_df.to_csv(os.path.join(OUTPUT_DIR, "feature_coefficients.csv"), index=False)

print("All outputs saved successfully!")
print("Output folder:", OUTPUT_DIR)
print("\nSaved files:")
print(os.listdir(OUTPUT_DIR))

All outputs saved successfully!
Output folder: C:\Users\Swathi\OneDrive\Desktop\Projects\Banking_Fraud_Analytics\ML_Outputs

Saved files:
['classification_report.json', 'feature_coefficients.csv', 'fraud_detection_model.joblib', 'test_predictions.csv', 'threshold_analysis.csv']


**Final Model Summary**

In [29]:
cm = confusion_matrix(y_test, y_pred)
tn, fp, fn, tp = cm.ravel()

print("FINAL FRAUD DETECTION SUMMARY")
print("-" * 35)
print("Test transactions:", len(y_test))
print("Actual fraud cases:", int(y_test.sum()))
print("True fraud detected:", tp)
print("Fraud missed:", fn)
print("Genuine transactions flagged as fraud:", fp)
print("Fraud Precision:", round(precision_score(y_test, y_pred, zero_division=0), 4))
print("Fraud Recall:", round(recall_score(y_test, y_pred, zero_division=0), 4))
print("Average Precision:", round(average_precision_score(y_test, y_prob), 4))
print("ROC-AUC:", round(roc_auc_score(y_test, y_prob), 4))

FINAL FRAUD DETECTION SUMMARY
-----------------------------------
Test transactions: 1293523
Actual fraud cases: 4258
True fraud detected: 3362
Fraud missed: 896
Genuine transactions flagged as fraud: 67265
Fraud Precision: 0.0476
Fraud Recall: 0.7896
Average Precision: 0.3168
ROC-AUC: 0.962


**Business impact**

In [30]:
flagged = int(y_pred.sum())
actual_fraud = int(y_test.sum())
detected = int(((y_test == 1) & (y_pred == 1)).sum())
missed = int(((y_test == 1) & (y_pred == 0)).sum())

print("BUSINESS IMPACT")
print("Total transactions:", len(y_test))
print("Transactions flagged:", flagged)
print("Flag rate:", round(flagged / len(y_test) * 100, 2), "%")
print("Alert precision:", round(detected / flagged * 100, 2), "%")
print("Fraud detection rate:", round(detected / actual_fraud * 100, 2), "%")
print("Fraud missed rate:", round(missed / actual_fraud * 100, 2), "%")

BUSINESS IMPACT
Total transactions: 1293523
Transactions flagged: 70627
Flag rate: 5.46 %
Alert precision: 4.76 %
Fraud detection rate: 78.96 %
Fraud missed rate: 21.04 %


**Best Threshold Comparison**

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score

threshold_df = pd.DataFrame([
    {
        "Threshold": t,
        "Precision": precision_score(y_test, y_prob >= t, zero_division=0),
        "Recall": recall_score(y_test, y_prob >= t, zero_division=0),
        "F1_Score": f1_score(y_test, y_prob >= t, zero_division=0)
    }
    for t in np.arange(0.1, 1.0, 0.1)
]).round(4)

print(threshold_df.to_string(index=False))

 Threshold  Precision  Recall  F1_Score
       0.1     0.0083  0.9937    0.0165
       0.2     0.0098  0.9847    0.0195
       0.3     0.0157  0.9403    0.0309
       0.4     0.0336  0.8567    0.0647
       0.5     0.0476  0.7896    0.0898
       0.6     0.0704  0.7489    0.1287
       0.7     0.1223  0.6595    0.2064
       0.8     0.1933  0.5207    0.2820
       0.9     0.2872  0.4016    0.3349
